# MedScope AI - Run Backend API on Colab

Run this notebook on Google Colab with a GPU (T4) to start the MedScope AI backend and expose it using `pyngrok`.

In [ ]:
# Clone the repository
!git clone https://github.com/muuo/finalyearproject.git
%cd finalyearproject

## 1. Install Dependencies

In [ ]:
!pip install -r backend/requirements.txt
!pip install -r models/requirements.txt
!pip install pyngrok qdrant-client

## 2. Setup Services (Redis, PostgreSQL, Qdrant)
Since Colab doesn't natively run Docker easily, we install the required services directly.

In [ ]:
# Install and start Redis and PostgreSQL
!apt-get update > /dev/null
!apt-get install -y redis-server postgresql > /dev/null
!service redis-server start
!service postgresql start

# Configure PostgreSQL database and user
!sudo -u postgres psql -c "CREATE USER medscope WITH PASSWORD 'password';"
!sudo -u postgres psql -c "CREATE DATABASE medscope_db OWNER medscope;"

# Download and run Qdrant vector database in the background
!wget -q https://github.com/qdrant/qdrant/releases/download/v1.7.0/qdrant-x86_64-unknown-linux-gnu.tar.gz
!tar -xzf qdrant-x86_64-unknown-linux-gnu.tar.gz
import subprocess
import time
subprocess.Popen(["./qdrant"])
time.sleep(3) # Wait for Qdrant to start

## 3. Run the Backend API
Enter your ngrok authtoken to expose the API to the public web so your frontend can communicate with it.

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import os

ngrok.set_auth_token("YOUR_NGROK_AUTH_TOKEN_HERE")

# Expose port 8000
public_url = ngrok.connect(8000)
print(f"\n\n======\nAPI is now available at: {public_url}\n======\n\n")

# Set custom weights path if available (if you trained models)
os.environ['YOLO_WEIGHTS_PATH'] = 'models/weights/yolov11_malaria_best.pt'

# Allow the nested asyncio loop in Colab to run Uvicorn
nest_asyncio.apply()

from backend.main import app
uvicorn.run(app, host="0.0.0.0", port=8000)